Step 2: Imports

In [1]:
import os
import re
import random
import contractions
import pandas as pd
import numpy as np

from langdetect import detect, DetectorFactory

from sklearn.model_selection import train_test_split

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

DetectorFactory.seed = 42

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Detect project root
current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, ".."))

if not os.path.exists(os.path.join(project_root, "data")):
    project_root = current_dir

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Step 2: Dataset Paths

In [2]:
dataset1_path = os.path.join(project_root, "data", "raw", "cross_platform_dataset.csv")
dataset2_path = os.path.join(project_root, "data", "raw", "kaggle_dataset.csv")
dataset3_path = os.path.join(project_root, "data", "raw", "twitter.csv")

Step 3: Load Datasets

In [3]:
df1 = pd.read_csv(dataset1_path)
df2 = pd.read_csv(dataset2_path)
df3 = pd.read_csv(dataset3_path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [4]:
print(df1.columns)
print(df2.columns)
print(df3.columns)

Index(['Platform', 'Text_Content', 'Target', 'Timestamp'], dtype='object')
Index(['tweet_text', 'cyberbullying_type'], dtype='object')
Index(['platform', 'text', 'label', 'Timestamp'], dtype='object')


Step 4: Harmonize Columns

In [5]:
# Dataset 1
df1.rename(
    columns={
        "Text_Content": "Text_Content",
        "Platform": "Platform",
        "Target": "Label"
    },
    inplace=True
)

df1["Label"] = df1["Label"].apply(
    lambda x: "Cyberbullying"
    if x == "Cyberbullying"
    else "not_cyberbullying"
)

# Dataset 2
df2.rename(
    columns={
        "tweet_text": "Text_Content",
        "cyberbullying_type": "Label"
    },
    inplace=True
)

df2["Platform"] = "Twitter"

df2["Label"] = df2["Label"].apply(
    lambda x:
    "not_cyberbullying"
    if x == "not_cyberbullying"
    else "Cyberbullying"
)

# Dataset 3
df3.rename(
    columns={
        "text": "Text_Content",
        "platform": "Platform",
        "label": "label"
    },
    inplace=True
)

df3["Label"] = df3["label"].apply(
    lambda x:
    "Cyberbullying"
    if x == 1
    else "not_cyberbullying"
)

print("Column names harmonized.")

Column names harmonized.


In [6]:
print(df1.columns)
print(df2.columns)
print(df3.columns)

Index(['Platform', 'Text_Content', 'Label', 'Timestamp'], dtype='object')
Index(['Text_Content', 'Label', 'Platform'], dtype='object')
Index(['Platform', 'Text_Content', 'label', 'Timestamp', 'Label'], dtype='object')


In [7]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

def is_english(text):
    try:
        if pd.isna(text):
            return False

        text = str(text).strip()

        if len(text) < 5:
            return False

        return detect(text) == "en"

    except:
        return False


print("Twitter dataset before filtering:", len(df2))

df2 = df2[df2["Text_Content"].apply(is_english)].reset_index(drop=True)

print("Twitter dataset after filtering:", len(df2))

Twitter dataset before filtering: 47692
Twitter dataset after filtering: 44659


Step 5: Improved Text Cleaning

In [8]:
slang_dict = {
    "u": "you",
    "ur": "your",
    "urs": "yours",
    "btw": "by the way",
    "idk": "i do not know",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "wtf": "what the fuck",
    "wth": "what the hell",
    "omg": "oh my god",
    "lmao": "laughing",
    "lol": "laughing",
    "rofl": "laughing",
    "brb": "be right back",
    "bcz": "because",
    "coz": "because",
    "cuz": "because",
    "pls": "please",
    "plz": "please",
    "thx": "thanks",
    "thnx": "thanks",
    "ty": "thank you",
    "smh": "shaking my head",
    "fyi": "for your information",
    "tbh": "to be honest",
    "irl": "in real life",
    "nvm": "never mind",
    "ikr": "i know right"
}


def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove @mentions
    text = re.sub(r'@\w+', ' ', text)

    # Remove hashtags completely
    text = re.sub(r'#\w+', ' ', text)

    # Expand contractions
    text = contractions.fix(text)

    # Normalize repeated letters (soooo -> soo)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Slang replacement
    words = [slang_dict.get(word, word) for word in text.split()]
    text = " ".join(words)

    # Remove everything except letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove single-letter words except "i" and "a"
    text = re.sub(r'\b(?![ia]\b)[a-z]\b', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Lemmatization
    text = " ".join(text.split())

    return text

Step 6: Apply Cleaning

In [9]:
# Apply cleaning
for dataset in [df1, df2, df3]:
    dataset["clean_text"] = dataset["Text_Content"].apply(clean_text)

print("Text preprocessing completed.")

# Remove empty tweets from training dataset
print("Twitter dataset before removing empty tweets:", len(df2))

df2 = df2[df2["clean_text"].str.strip() != ""].reset_index(drop=True)

print("Twitter dataset after removing empty tweets:", len(df2))

Text preprocessing completed.
Twitter dataset before removing empty tweets: 44659
Twitter dataset after removing empty tweets: 44566


In [10]:
print(df2["clean_text"].head(10))

0            in other words your food was crapilicious
1                                      why is so white
2           a classy whore or more red velvet cupcakes
3    meh thanks for the heads up but not too concer...
4    this is an isis account pretending to be a kur...
5    yes the test of god is that good or bad or ind...
6    karma i hope it bites kat on the butt she is j...
7                      everything but mostly my priest
8    rebecca black drops out of school due to bullying
9                              the bully flushes on kd
Name: clean_text, dtype: object


Step 7: Encode Labels

In [11]:
label_mapping = {
    "Cyberbullying": 1,
    "not_cyberbullying": 0
}

for dataset in [df1, df2, df3]:
    dataset["label_encoded"] = dataset["Label"].map(label_mapping)

print("Labels encoded.")

Labels encoded.


In [12]:
# Find texts with conflicting labels
conflicts = (
    df2.groupby("clean_text")["label_encoded"]
       .nunique()
)

conflicting_texts = conflicts[conflicts > 1].index

print("Conflicting texts:", len(conflicting_texts))

# Remove all conflicting texts
df2 = df2[~df2["clean_text"].isin(conflicting_texts)].reset_index(drop=True)

print("Dataset after removing conflicting texts:", len(df2))

# Remove remaining exact duplicate tweet-label pairs
df2 = df2.drop_duplicates(
    subset=["clean_text", "label_encoded"]
).reset_index(drop=True)

print("Dataset after removing duplicates:", len(df2))

Conflicting texts: 1345
Dataset after removing conflicting texts: 41796
Dataset after removing duplicates: 41142


Step 8: Shuffle Twitter Dataset

In [13]:
df2 = df2.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

duplicate removal before splitting

In [14]:
print("Checking duplicate tweets...")

duplicate_count = df2["clean_text"].duplicated().sum()

print(f"Duplicate tweets found: {duplicate_count}")

Checking duplicate tweets...
Duplicate tweets found: 0


In [15]:
print("Twitter dataset before removing duplicates:", len(df2))

# Remove exact duplicate tweet-label pairs only
df2 = df2.drop_duplicates(
    subset=["clean_text", "label_encoded"]
).reset_index(drop=True)

print("Twitter dataset after removing duplicates:", len(df2))

Twitter dataset before removing duplicates: 41142
Twitter dataset after removing duplicates: 41142


Step 9: Train / Validation / Test Split

In [16]:
# 80% Train+Validation | 20% Twitter Test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    df2["clean_text"],
    df2["label_encoded"],
    test_size=0.20,
    stratify=df2["label_encoded"],
    random_state=SEED
)

# 10% Validation (from training data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.125,
    stratify=y_train_full,
    random_state=SEED
)

# Independent Multi-platform Test
X_test_cross = df1["clean_text"]
y_test_cross = df1["label_encoded"]

# Live Demo
X_live_demo = df3["clean_text"]
y_live_demo = df3["label_encoded"]

print("Dataset splitting completed.")

Dataset splitting completed.


In [17]:
# Check exact duplicate text between Train and Test
train_set = set(X_train)
test_set = set(X_test)
cross_set = set(X_test_cross)

print(f"Duplicates between Train & Test: {len(train_set.intersection(test_set))}")
print(f"Duplicates between Train & Cross-Test: {len(train_set.intersection(cross_set))}")

Duplicates between Train & Test: 0
Duplicates between Train & Cross-Test: 0


Step 10: Save Train / Validation / Test Splits

In [18]:
processed_folder = os.path.join(
    project_root,
    "data",
    "processed",
    "split"
)

os.makedirs(processed_folder, exist_ok=True)

X_train.to_csv(os.path.join(processed_folder, "X_train.csv"), index=False)
y_train.to_csv(os.path.join(processed_folder, "y_train.csv"), index=False)

X_val.to_csv(os.path.join(processed_folder, "X_val.csv"), index=False)
y_val.to_csv(os.path.join(processed_folder, "y_val.csv"), index=False)

X_test.to_csv(os.path.join(processed_folder, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(processed_folder, "y_test.csv"), index=False)

X_test_cross.to_csv(os.path.join(processed_folder, "X_test_cross.csv"), index=False)
y_test_cross.to_csv(os.path.join(processed_folder, "y_test_cross.csv"), index=False)

X_live_demo.to_csv(os.path.join(processed_folder, "X_live_demo.csv"), index=False)
y_live_demo.to_csv(os.path.join(processed_folder, "y_live_demo.csv"), index=False)

# Platform-aware cross-platform file
df1.to_csv(
    os.path.join(processed_folder, "X_test_cross_full.csv"),
    index=False
)

print("All dataset splits saved.")

All dataset splits saved.


Step 11: Save Processed Datasets

In [19]:
processed_dataset_folder = os.path.join(
    project_root,
    "data",
    "processed"
)

os.makedirs(processed_dataset_folder, exist_ok=True)

df1.to_csv(
    os.path.join(processed_dataset_folder, "cross_platform_dataset_processed.csv"),
    index=False
)

df2.to_csv(
    os.path.join(processed_dataset_folder, "kaggle_dataset_processed.csv"),
    index=False
)

df3.to_csv(
    os.path.join(processed_dataset_folder, "twitter_api_processed.csv"),
    index=False
)

print("Processed datasets saved.")

Processed datasets saved.


Step 12: Data Quality Checks

In [20]:
for name, dataset in zip(
    ["Dataset 1", "Dataset 2", "Dataset 3"],
    [df1, df2, df3]
):

    print(f"\n{name}")

    print("- Missing Values")
    print(dataset[["clean_text", "label_encoded"]].isnull().sum())

    print("\n- Duplicate Texts")
    print(dataset["clean_text"].duplicated().sum())

    print("\n- Empty Texts")
    print((dataset["clean_text"] == "").sum())

    print("\n- Class Distribution")
    print(dataset["label_encoded"].value_counts())

    print("\n- Sample Records")
    print(dataset[["clean_text", "label_encoded"]].sample(5, random_state=SEED))


Dataset 1
- Missing Values
clean_text       0
label_encoded    0
dtype: int64

- Duplicate Texts
2238

- Empty Texts
0

- Class Distribution
label_encoded
0    1173
1    1085
Name: count, dtype: int64

- Sample Records
                        clean_text  label_encoded
2151     you are annoying everyone              1
368   that was an interesting read              0
527    i hate everything about you              1
1047             good job everyone              0
56          just disappear already              1

Dataset 2
- Missing Values
clean_text       0
label_encoded    0
dtype: int64

- Duplicate Texts
0

- Empty Texts
0

- Class Distribution
label_encoded
1    35945
0     5197
Name: count, dtype: int64

- Sample Records
                                              clean_text  label_encoded
4750   people say i am a public figure but eww twitte...              1
29141     just in case anyone is down for some meatballs              1
29897  that is why you were my first friend o

In [21]:
for name, dataset in zip(
    ["Dataset 1", "Dataset 2", "Dataset 3"],
    [df1, df2, df3]
):
    ...
    print(dataset["clean_text"].duplicated().sum())

2238
0
0


In [22]:
print("Dataset 1 duplicates:", df1["clean_text"].duplicated().sum())
print("Dataset 2 duplicates:", df2["clean_text"].duplicated().sum())
print("Dataset 3 duplicates:", df3["clean_text"].duplicated().sum())

Dataset 1 duplicates: 2238
Dataset 2 duplicates: 0
Dataset 3 duplicates: 0
